# Labyrinth Core

Labyrinth Core prepares a quantum graph state: a set of qubits where you choose how each one points and how each connected pair correlates, then samples it. Pairs set to agree tend to measure the same bit; pairs set to disagree, opposite bits. Labyrinth Level builds its mazes on this: every room a qubit, every corridor a pair that agrees. (The API names it Quantum Graph.)

## Setup

In [ ]:
import getpass
import os

import matplotlib.pyplot as plt

from moth_atlas import Atlas

ENGINE = "graph-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "graph-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

In [ ]:
def find(tree, name):
    """The first value under the key `name` anywhere in a JSON result (an exact key first, else the first key
    containing `name`), or None. The engine nests its numbers in sections; this pulls one out by name."""
    def walk(node, match):
        if isinstance(node, dict):
            for key, value in node.items():
                if match(key):
                    return value
            node = list(node.values())
        if isinstance(node, list):
            for value in node:
                if (found := walk(value, match)) is not None:
                    return found
        return None
    found = walk(tree, lambda key: key == name)
    return found if found is not None else walk(tree, lambda key: name in str(key))


def counts(job):
    """The measured bitstrings and how often each came up."""
    found = find(job.result, "counts")
    if not isinstance(found, dict):
        raise KeyError(f"no counts in this result; its sections are: {list(job.result)}")
    return found


def bit(bitstring, qubit):
    """One qubit's bit. Qiskit order: qubit 0 is the rightmost character."""
    return bitstring[::-1][qubit]


def agreement(job, pairs):
    """For each pair, the share of shots where both qubits gave the same bit."""
    c = counts(job)
    total = sum(c.values())
    return {tuple(p): sum(n for s, n in c.items() if bit(s, p[0]) == bit(s, p[1])) / total for p in pairs}


def histogram(job, top=12):
    """The most frequent outcomes, as a bar chart."""
    c = sorted(counts(job).items(), key=lambda kv: -kv[1])[:top]
    plt.figure(figsize=(8, 2))
    plt.bar([s for s, _ in c], [n for _, n in c])
    plt.xticks(rotation=60, fontsize=8)
    plt.ylabel("shots")
    plt.show()

## Try it

Four qubits in a line, like four rooms in a corridor. Qubit 0 is pointed along X, then each neighbouring pair gets a target: 0–1 agree, 1–2 disagree, 2–3 agree. The outcomes should mostly be the two patterns that fit: a wall between rooms 1 and 2.

In [ ]:
chain = [[0, 1], [1, 2], [2, 3]]

job = client.jobs.run(
    engine=ENGINE,
    params={
        "num_qubits": 4,
        "coupling_map": chain,
        "operations": [
            {"type": "bloch", "qubit": 0, "paulis": {"X": 1.0}},
            {"type": "relationship", "qubits": [0, 1], "paulis": {"ZZ": 1.0}},
            {"type": "relationship", "qubits": [1, 2], "paulis": {"ZZ": -1.0}},
            {"type": "relationship", "qubits": [2, 3], "paulis": {"ZZ": 1.0}},
        ],
        "shots": 1024,
        "mode": "emu",
    },
)
histogram(job)
for pair, share in agreement(job, chain).items():
    print(f"qubits {pair}: same bit in {share:.0%} of shots")

## Parameters

What to expect from each:

- `num_qubits` (2 to 20): the graph's nodes.
- `coupling_map`: the edges, as qubit pairs. Leave it out for a fully connected graph.
- `operations`: targets applied in order. `bloch` points one qubit (`{"X": 1.0}`, `{"Z": -1.0}`, …); `relationship` sets a coupled pair's correlation (`{"ZZ": 1.0}` agree, `{"ZZ": -1.0}` disagree, or `XX`, `YY`, …). Each takes a `fraction` of the full rotation (default 1) and `update` (default true: re-measure the state so later operations see earlier ones). Leave `operations` out for a random state.
- `seed`: fixes the random graph when there are no operations; the same seed builds the same circuit.
- `shots`: how many samples.
- `mode`: `emu` simulates without noise, `fake` replays a real chip's noise (with `backend_name`), `qpu` runs on IBM hardware (your account needs access).

In [ ]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Targets, not gates.** You say what you want the state to look like: single-qubit expectation values such as $\langle X_0\rangle = 1$, or pair correlations such as $\langle Z_1 Z_2\rangle = -1$. The engine (built on the QuantumGraph library) works out the rotations that move the current state towards each target, one operation at a time, using tomography of the state so far.

**Sample.** The prepared state is measured `shots` times. For a pair with $\langle Z_iZ_j\rangle = c$, the share of shots where both bits agree is $(1 + c)/2$: 100% for $c = 1$, 0% for $c = -1$, 50% for no correlation.

**Why it's quantum.** The targets are reached together in one entangled state over $2^n$ dimensions; a pair can be fully correlated while each qubit on its own is a coin flip. On hardware, noise loosens the correlations, and that loosening is part of what Labyrinth's mazes show.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Correlation target

Two qubits, one `ZZ` target from −1 to 1: the share of agreeing shots should follow $(1 + c)/2$.

In [ ]:
targets, measured = (-1.0, -0.5, 0.0, 0.5, 1.0), []
for zz in targets:
    print(f"ZZ={zz}")
    job = client.jobs.run(engine=ENGINE, params={
        "num_qubits": 2, "coupling_map": [[0, 1]], "shots": 2048,
        "operations": [{"type": "bloch", "qubit": 0, "paulis": {"X": 1.0}},
                       {"type": "relationship", "qubits": [0, 1], "paulis": {"ZZ": zz}}]}, quiet=True)
    measured.append(agreement(job, [[0, 1]])[(0, 1)])
plt.figure(figsize=(4, 3))
plt.plot(targets, [(1 + c) / 2 for c in targets], "k--", label="(1 + c) / 2")
plt.plot(targets, measured, "o", label="measured")
plt.xlabel("ZZ target c")
plt.ylabel("same bit")
plt.legend()
plt.show()

### Fraction

`fraction` applies part of the rotation towards a target: a softer, partial correlation.

In [ ]:
for fraction in (0.25, 0.5, 0.75, 1.0):
    job = client.jobs.run(engine=ENGINE, params={
        "num_qubits": 2, "coupling_map": [[0, 1]], "shots": 2048,
        "operations": [{"type": "bloch", "qubit": 0, "paulis": {"X": 1.0}},
                       {"type": "relationship", "qubits": [0, 1], "paulis": {"ZZ": 1.0}, "fraction": fraction}]}, quiet=True)
    print(f"fraction={fraction}: same bit in {agreement(job, [[0, 1]])[(0, 1)]:.0%} of shots")

### Random graphs

With no operations, the engine draws a random state from `seed`: the same seed gives the same circuit, so its outcome frequencies match up to shot noise.

In [ ]:
for seed in (1, 1, 2):
    print(f"seed={seed}")
    histogram(client.jobs.run(engine=ENGINE, params={"num_qubits": 5, "seed": seed, "shots": 2048}, quiet=True), top=8)

## Questions

**How does this make a maze?**

Labyrinth Level (`labyrinth/`) lays rooms on a grid, one qubit each, couples neighbouring rooms, and sets a `ZZ` target per edge from the level design. Neighbours that measure the same bit are joined by a corridor; ones that differ get a wall.

**Why don't I get exactly the targets?**

Shots: every share is estimated from a finite sample. Targets can also conflict (three pairs in a triangle can't all disagree); later operations then pull against earlier ones.

**Does it run on hardware?**

`mode: "qpu"` submits to IBM hardware (your account needs access); `fake` replays a chip's noise. Up to 20 qubits.